# Export fragment files to be processed by chrombpnet

In [1]:
suppressPackageStartupMessages(library(ArchR))
suppressPackageStartupMessages(library(parallel))



                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ==='      `-'    | ##,-/
        -,____,---'       \\####\\________________,--\\_##,/
           ___      .______        ______  __    __  .______      
          /   \     |   _ 

In [2]:
here::i_am("atac/chrombpnet/01_export_fragments.ipynb")

source(here::here("settings.R"))
source(here::here("utils.R"))


here() starts at /rds/project/rds-SDzz0CATGms/users/bt392/09_Eomes_invitro_blood/code

Warning message:
“package ‘rtracklayer’ was built under R version 4.2.3”


In [3]:
ArchRProject = loadArchRProject(io$archR.directory)
meta = fread(paste0(io$basedir, '/results/rna_atac/clustering/metadata_celltype_annotated_v2.txt.gz'))

Successfully loaded ArchRProject!


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ==='      `-'    | ##,-/
        -,____,---'       \\####\\________________,--\\_##,/
           ___      .______        ______  __    __  .____

In [4]:
outdir = paste0(io$basedir, '/results/atac/chrombpnet/')
dir.create(outdir, recursive = T)

Warning message in dir.create(outdir, recursive = T):
“'/rds/project/rds-SDzz0CATGms/users/bt392/09_Eomes_invitro_blood/results/atac/chrombpnet' already exists”


In [5]:
meta_WT = meta %>% .[genotype == 'WT']

In [6]:
unique(meta_WT$celltype_v2)

[1] "Primitive_Streak"    "Early_Mes_EOi"       "Early_Mes_EOd"      
 [4] "PGC"                 "Posterior_Mes"       "HE_Precursor"       
 [7] "HE"                  "Mesenchyme"          "Allantois_Precursor"
[10] "Blood_Progenitor"    "Endothelium"         "Allantois"

In [20]:
chrs = c('chr1','chr10','chr11','chr12','chr13','chr14','chr15','chr16','chr17','chr18','chr19','chr2','chr3','chr4','chr5','chr6','chr7','chr8','chr9')
# exclude samples
# samples_incl = unique(meta$sample)[!unique(meta$sample) %in% c('E8.5_CRISPR_T_KO','E8.5_CRISPR_T_WT')]
# meta = meta[sample %in% samples_incl]

# Cell types of interest
celltypes_keep = unique(meta_WT$celltype_v2)

# Export fragments of celltype cells in subset of samples
mclapply(unique(meta_WT$sample), function(x){
    # List cells per sample
    message(x)
    # filter right samples
    tmp_meta = meta[sample == x]  
    # Keep relevant cell types
    celltypes = tmp_meta %>%
        .[, .N, by = 'celltype_v2'] %>% 
        unique(by = 'celltype_v2') %>%
        .[N > 50] %>% 
        .$celltype_v2
    
    celltypes = celltypes[celltypes %in% celltypes_keep]
    
    # Load fragment file
    if(!all(file.exists(sprintf('%s/fragments/%s_%s.tsv.gz', outdir, celltypes, x)))){
        file = sprintf('%s/original/%s/outs/atac_fragments.tsv.gz', io$basedir, x)
        fragment = suppressWarnings(fread(file, nThread = 38,
                                            tmpdir = '/rds/project/rds-SDzz0CATGms/users/bt392/software/tmp'))
        message('before filtering:')
        message(nrow(fragment))
       
        mclapply(celltypes, function(i){
            message(i)
            # Only run if file doesn't exist yet
            if(!file.exists(sprintf('%s/fragments/%s_%s.tsv.gz', outdir, i, x))){
                # determine cells
                tmp_cell = tmp_meta[celltype_v2 == i, barcode]

                # Filter right cells & chrs from fragment file
                fragment = fragment %>% 
                    setnames(c('chr', 'start', 'end', 'cell', 'reads')) %>%
                    # .[chr == 'chr1'] %>%
                    .[cell %in% tmp_cell] %>% 
                    .[chr %in% chrs]

                message('after filtering:')
                message(nrow(fragment))

                fwrite(fragment, sprintf('%s/fragments/%s_%s.tsv.gz', outdir, i, x), sep = '\t', nThread = 38)
            }
        }, mc.cores = 4)
    }else{message('All fragments already exist')}

}, mc.cores = 1)

1A_Eo_DEG_G9_day3

All fragments already exist

1B_Eo_DEG_G9_day3

All fragments already exist

2_Eo_DEG_G9_day3_5_VC

before filtering:

301618929

2_Eo_DEG_G9_day4_VC

before filtering:

184863381

2_Eo_DEG_G9_day5_VC

before filtering:

32380584

rv_eo_deg_day3_5_control

before filtering:

234284379

rv_eo_deg_day4_5_control

before filtering:

73281386

rv_eo_deg_day4_control

before filtering:

176167120



[[1]]
NULL

[[2]]
NULL

[[3]]
[[3]][[1]]
NULL

[[3]][[2]]
NULL

[[3]][[3]]
NULL

[[3]][[4]]
NULL

[[3]][[5]]
NULL

[[3]][[6]]
NULL


[[4]]
[[4]][[1]]
NULL

[[4]][[2]]
NULL

[[4]][[3]]
NULL

[[4]][[4]]
NULL

[[4]][[5]]
NULL

[[4]][[6]]
NULL

[[4]][[7]]
NULL


[[5]]
[[5]][[1]]
NULL

[[5]][[2]]
NULL

[[5]][[3]]
NULL

[[5]][[4]]
NULL


[[6]]
[[6]][[1]]
NULL

[[6]][[2]]
NULL

[[6]][[3]]
NULL

[[6]][[4]]
NULL

[[6]][[5]]
NULL

[[6]][[6]]
NULL


[[7]]
[[7]][[1]]
NULL

[[7]][[2]]
NULL

[[7]][[3]]
NULL

[[7]][[4]]
NULL

[[7]][[5]]
NULL

[[7]][[6]]
NULL


[[8]]
[[8]][[1]]
NULL

[[8]][[2]]
NULL

[[8]][[3]]
NULL

[[8]][[4]]
NULL

In [8]:
getDTthreads()

[1] 1

In [10]:
celltypes_keep = unique(meta_WT$celltype_v2)

lapply(celltypes_keep, function(i){    
    message(i)
    if(!file.exists(sprintf('%s/fragments/%s_fragments.tsv.gz', outdir, i))){
        # List files per cell type
        files = paste0(i, '_', unique(meta_WT$sample), '.tsv.gz')
        files = files[files %in% list.files(file.path(outdir, 'fragments'), pattern = i)]
        
        message(files)
        # Load in the separate fragment files per cell type
        tmp = mclapply(files, function(x){
            tmp_frag = fread(sprintf('%s/%s', file.path(outdir, 'fragments'), x), sep = '\t', nThread = 12,
                                            tmpdir = '/rds/project/rds-SDzz0CATGms/users/bt392/software/tmp')
            return(tmp_frag)
        }, mc.cores = 8) %>% rbindlist() %>% 
        .[order(chr, start)] %>% 
        .[,reads := 1]

        fwrite(tmp, sprintf('%s/fragments/%s_fragments.tsv.gz', outdir, i), col.names = F, sep = '\t', nThread = 38)
    }
})

Primitive_Streak

Early_Mes_EOi

Early_Mes_EOd

Early_Mes_EOd_2_Eo_DEG_G9_day3_5_VC.tsv.gzEarly_Mes_EOd_rv_eo_deg_day3_5_control.tsv.gz

PGC

PGC_2_Eo_DEG_G9_day3_5_VC.tsv.gzPGC_2_Eo_DEG_G9_day4_VC.tsv.gzPGC_rv_eo_deg_day3_5_control.tsv.gz

Posterior_Mes

Posterior_Mes_2_Eo_DEG_G9_day4_VC.tsv.gzPosterior_Mes_rv_eo_deg_day4_control.tsv.gz

HE_Precursor

HE_Precursor_2_Eo_DEG_G9_day3_5_VC.tsv.gzHE_Precursor_rv_eo_deg_day3_5_control.tsv.gz

HE

HE_2_Eo_DEG_G9_day3_5_VC.tsv.gzHE_2_Eo_DEG_G9_day4_VC.tsv.gzHE_rv_eo_deg_day3_5_control.tsv.gzHE_rv_eo_deg_day4_5_control.tsv.gzHE_rv_eo_deg_day4_control.tsv.gz

Mesenchyme

Mesenchyme_2_Eo_DEG_G9_day4_VC.tsv.gzMesenchyme_2_Eo_DEG_G9_day5_VC.tsv.gzMesenchyme_rv_eo_deg_day4_5_control.tsv.gzMesenchyme_rv_eo_deg_day4_control.tsv.gz

Allantois_Precursor

Allantois_Precursor_2_Eo_DEG_G9_day4_VC.tsv.gzAllantois_Precursor_rv_eo_deg_day4_5_control.tsv.gzAllantois_Precursor_rv_eo_deg_day4_control.tsv.gz

Blood_Progenitor

Blood_Progenitor_2_Eo_DEG_G9_day5_V

[[1]]
NULL

[[2]]
NULL

[[3]]
NULL

[[4]]
NULL

[[5]]
NULL

[[6]]
NULL

[[7]]
NULL

[[8]]
NULL

[[9]]
NULL

[[10]]
NULL

[[11]]
NULL

[[12]]
NULL

In [24]:
peaks = as.data.table(ArchRProject@peakSet) %>%
    .[, middle := start + (end - start) / 2] %>%
    .[,`:=`(start = middle - 1057, 
            end = middle + 1057,
            column4 = '.',
            column5 = '.',
            column6 = '.',
            column7 = '.',
            column8 = '.',
            column9 = '.',
            column10 = 1057)] %>% 
    .[,.(seqnames, 
         start, 
         end, 
         column4,
         column5,
         column6,
         column7,
         column8,
         column9,
         column10)]

nrow(peaks)
head(peaks)

[1] 234908

seqnames,start,end,column4,column5,column6,column7,column8,column9,column10
<fct>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>
chr1,3002713,3004827,.,.,.,.,.,.,1057
chr1,3034841,3036955,.,.,.,.,.,.,1057
chr1,3061882,3063996,.,.,.,.,.,.,1057
chr1,3190830,3192944,.,.,.,.,.,.,1057
chr1,3262833,3264947,.,.,.,.,.,.,1057
chr1,3482191,3484305,.,.,.,.,.,.,1057


In [25]:
peaks.gr = GenomicRanges::makeGRangesFromDataFrame(peaks, keep.extra.columns = T)
overlaps = GenomicRanges::findOverlaps(peaks.gr, ArchRProject@genomeAnnotation$blacklist)

peaks.gr = peaks.gr[-queryHits(overlaps)]

In [26]:
peaks = as.data.table(peaks.gr)[,`:=`(width=NULL, strand=NULL)]
fwrite(peaks, sprintf('%s/all_peaks.bed', outdir), col.names = F, sep = '\t')

In [27]:
chromSizes = as.data.table(ArchRProject@genomeAnnotation$chromSizes) %>%
    .[,.(seqnames, end)]
fwrite(chromSizes, sprintf('%s/mm10.chrom.sizes', outdir), col.names = F, sep = '\t')

In [28]:
fwrite(as.data.table(ArchRProject@genomeAnnotation$blacklist) %>%
           .[,.(seqnames, start, end)],
       sprintf('%s/blacklist.bed.gz', outdir), col.names = F, sep = '\t')

In [12]:
peaks = fread(sprintf('%s/all_peaks.bed', outdir))

In [30]:
peaks_matrix =  peaks %>% 
    .[, middle := V2 + (V3 - V2) / 2] %>%
    .[,`:=`(start = middle + 1057, 
            end = middle - 1057)]

In [31]:
head(peaks_matrix)

V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,middle,start,end
<chr>,<int>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<dbl>,<dbl>,<dbl>
chr1,3002713,3004827,.,.,.,.,.,.,1057,3003770,3004827,3002713
chr1,3034841,3036955,.,.,.,.,.,.,1057,3035898,3036955,3034841
chr1,3061882,3063996,.,.,.,.,.,.,1057,3062939,3063996,3061882
chr1,3190830,3192944,.,.,.,.,.,.,1057,3191887,3192944,3190830
chr1,3262833,3264947,.,.,.,.,.,.,1057,3263890,3264947,3262833
chr1,3482191,3484305,.,.,.,.,.,.,1057,3483248,3484305,3482191


In [38]:
peaks = as.data.table(ArchRProject@peakSet) %>%
    .[, middle := start + (end - start) / 2] %>%
    .[,`:=`(start = middle - 1057, 
            end = middle + 1057,
            column4 = .I,
            column5 = '.',
            column6 = '.',
            column7 = '.',
            column8 = '.',
            column9 = '.',
            column10 = 1057)] %>% 
    .[,.(seqnames, 
         start, 
         end, 
         column4,
         column5,
         column6,
         column7,
         column8,
         column9,
         column10)]

nrow(peaks)
head(peaks)

[1] 234908

seqnames,start,end,column4,column5,column6,column7,column8,column9,column10
<fct>,<dbl>,<dbl>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>
chr1,3002713,3004827,1,.,.,.,.,.,1057
chr1,3034841,3036955,2,.,.,.,.,.,1057
chr1,3061882,3063996,3,.,.,.,.,.,1057
chr1,3190830,3192944,4,.,.,.,.,.,1057
chr1,3262833,3264947,5,.,.,.,.,.,1057
chr1,3482191,3484305,6,.,.,.,.,.,1057


In [39]:
peaks.gr = GenomicRanges::makeGRangesFromDataFrame(peaks, keep.extra.columns = T)
overlaps = GenomicRanges::findOverlaps(peaks.gr, ArchRProject@genomeAnnotation$blacklist)

peaks.gr = peaks.gr[-queryHits(overlaps)]

In [40]:
peaks = as.data.table(peaks.gr)[,`:=`(width=NULL, strand=NULL)]

In [20]:
atac.sce = getMatrixFromProject(
  ArchRProj = ArchRProject,
  useMatrix = "PeakMatrix",
  verbose = FALSE,
  binarize = FALSE,
  threads = getArchRThreads()
)

ArchR logging to : ArchRLogs/ArchR-getMatrixFromProject-322fcd225a5a26-Date-2025-03-13_Time-15-32-30.log
If there is an issue, please report to github with logFile!



In [44]:
# Only keep peaks that do not overlap with blacklisted regions
atac.sce = atac.sce[peaks$column4,]

In [47]:
atac.sce

class: RangedSummarizedExperiment 
dim: 234426 63114 
metadata(0):
assays(1): PeakMatrix
rownames: NULL
rowData names(1): idx
colnames(63114): 2_Eo_DEG_G9_day3_5_VC#CACCTCAGTTACTTCA-1
  2_Eo_DEG_G9_day3_5_VC#GGGTTATTCGATTTGA-1 ...
  2_Eo_DEG_G9_day5_VC#GTAGCTGTCCGGCTAA-1
  2_Eo_DEG_G9_day5_VC#GGTCTTTGTTTCAGGA-1
colData names(39): BlacklistRatio nDiFrags ... FRIP replicate

In [48]:
atac.rse

class: RangedSummarizedExperiment 
dim: 234908 63114 
metadata(0):
assays(1): PeakMatrix
rownames: NULL
rowData names(0):
colnames(63114): 2_Eo_DEG_G9_day3_5_VC#CACCTCAGTTACTTCA-1
  2_Eo_DEG_G9_day3_5_VC#GGGTTATTCGATTTGA-1 ...
  2_Eo_DEG_G9_day5_VC#GTAGCTGTCCGGCTAA-1
  2_Eo_DEG_G9_day5_VC#GGTCTTTGTTTCAGGA-1
colData names(39): BlacklistRatio nDiFrags ... FRIP replicate

In [49]:
head(peaks)

seqnames,start,end,column4,column5,column6,column7,column8,column9,column10
<fct>,<int>,<int>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>
chr1,3002713,3004827,1,.,.,.,.,.,1057
chr1,3034841,3036955,2,.,.,.,.,.,1057
chr1,3061882,3063996,3,.,.,.,.,.,1057
chr1,3190830,3192944,4,.,.,.,.,.,1057
chr1,3262833,3264947,5,.,.,.,.,.,1057
chr1,3482191,3484305,6,.,.,.,.,.,1057


In [45]:
library(BSgenome.Mmusculus.UCSC.mm10)

In [52]:
counts(atac.sce) = atac.sce@assays@data$PeakMatrix

In [53]:
## Calculate background peaks
cat('Calculating background peaks \n')
gr <- GRanges(
    seqnames = Rle(peaks$seqnames),
    ranges = IRanges(start = peaks$start, 
                     end = peaks$end),
    strand = Rle(rep('*', nrow(peaks))))
atac.rse = as(atac.sce, 'RangedSummarizedExperiment')
atac.rse@rowRanges = gr

## Adding background peaks
atac.rse <- chromVAR::addGCBias(atac.rse, genome = BSgenome.Mmusculus.UCSC.mm10)
bg <- chromVAR::getBackgroundPeaks(object = atac.rse)

Calculating background peaks 
